In [ ]:
include("../src/lattice.jl")
include("../src/metropolis_ising.jl")
using .UnitCell_Lattices
using .Metropolis_MonteCarlo

using Plots
using StatsBase
using LsqFit
using StatsPlots

# Simulation of the Ferromagnetic Ising Model on Square Lattice

In [2]:
# Building the square lattice
a = 1.0

A = [a*1.0 0.0;
     0.0 a*1.0] # Lattice vectors for a triangular lattice

B = [0.0;
     0.0] #Basis vectors
B = reshape(B, (2,1)) #Converting Basis vectors into a Matrix as required by the module
     
uc_square = UnitCell(A,B)
rules_square = [    BondRule(1,1,[1,0]),
    BondRule(1,1,[0,1])]

2-element Vector{BondRule}:
 BondRule(1, 1, [1, 0])
 BondRule(1, 1, [0, 1])

In [28]:
# Defining thermodynamic variables
J = 1.0
h = 0.0

T = 0.2:0.1:4.0
L_list = [[4,4],[12,12],[24,24],[64,64]]

4-element Vector{Vector{Int64}}:
 [4, 4]
 [12, 12]
 [24, 24]
 [64, 64]

In [29]:
# Getting the data for each temperature
data = Dict()
n_sweep = 200000
n_thermal = 50000
n_skip = 40
for L in L_list
    for Temp in T
        lattice = build_lattice(uc_square,rules_square,L)
        for site in lattice.sites
            site.spin = [0,0,rand([-1,1])]
        end

        res = monte_carlo_simulation(lattice,Temp,J,h,n_sweep,n_thermal,n_skip)
        data[(L,Temp)] = (energies = res[1], magnetization = res[2], magnetization2 = res[3], magnetization4 = res[2].^4)
    end
end

In [30]:
using JLD2
@save "ising_square_data.jld2" data
data[([4,4],3.1)]

(energies = [-48.0, -48.0, -64.0, -32.0, -16.0, -8.0, -32.0, -64.0, -16.0, -48.0  …  -32.0, -32.0, -32.0, -8.0, -16.0, -16.0, -8.0, 0.0, -24.0, -8.0], magnetization = [0.875, -0.875, -1.0, 0.625, -0.25, -0.125, -0.625, 1.0, 0.25, 0.875  …  0.75, -0.625, 0.75, 0.25, 0.125, 0.125, -0.375, 0.375, 0.5, -0.25], magnetization2 = [0.765625, 0.765625, 1.0, 0.390625, 0.0625, 0.015625, 0.390625, 1.0, 0.0625, 0.765625  …  0.5625, 0.390625, 0.5625, 0.0625, 0.015625, 0.015625, 0.140625, 0.140625, 0.25, 0.0625], magnetization4 = [0.586181640625, 0.586181640625, 1.0, 0.152587890625, 0.00390625, 0.000244140625, 0.152587890625, 1.0, 0.00390625, 0.586181640625  …  0.31640625, 0.152587890625, 0.31640625, 0.00390625, 0.000244140625, 0.000244140625, 0.019775390625, 0.019775390625, 0.0625, 0.00390625])